# EMNIST: финальный буст для v5 — 50 эпох + Mixup

**Цель:** получить «чемпионскую» точность для аннотации диссертации.

**Конфигурация:**
- Модель: improved_cnn_v5 (SE-CNN, 712K параметров)
- Аугментация: light
- Scheduler: cosine
- **Эпохи: 50** (вместо 25)
- **Mixup α=0.2** (новое)
- AdamW + label_smoothing=0.1
- seed=42, batch=64, lr=1e-3

**Ожидаемое время:** ~25–35 мин на T4.

**Базовый результат для сравнения:** v5_light_cosine = 91,15% (без Mixup, 25 эпох).  
**Ожидаемый результат с бустом:** 91,5–91,8%.

## 1. Проверка GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ"}')
!nvidia-smi

## 2. Клонирование репозитория

In [ ]:
REPO_URL = 'https://github.com/sultonovmuhibulloh612/emnist-master-thesis.git'

import os
os.chdir('/kaggle/working')

if not os.path.exists('/kaggle/working/emnist-project'):
    !git clone $REPO_URL /kaggle/working/emnist-project
%cd /kaggle/working/emnist-project

!pip install -q tensorboard seaborn

## 3. Проверка наличия Mixup в train.py

**Перед запуском убедись, что в твоём `train.py` уже добавлен Mixup** (см. инструкцию выше).

In [ ]:
with open('src/training/train.py', encoding='utf-8') as f:
    code = f.read()

checks = {
    'AdamW в оптимизаторе':       'optim.AdamW' in code,
    'label_smoothing=0.1':         'label_smoothing=0.1' in code,
    'v5 в реестре':                'improved_cnn_v5' in code,
    'параметр --mixup_alpha':      '--mixup_alpha' in code,
    'логика Mixup в train_one_epoch': 'mixup_alpha' in code and 'np.random.beta' in code,
    'import numpy as np':          'import numpy as np' in code,
}
for desc, ok in checks.items():
    print(f'{"✓" if ok else "✗"} {desc}')

if not all(checks.values()):
    print('\n⚠ ВАЖНО: добавь недостающие части в train.py перед запуском!')
else:
    print('\n✓ Всё готово к запуску')

## 4. Предзагрузка EMNIST

In [ ]:
from torchvision import datasets, transforms

DATA_DIR = '/kaggle/working/emnist-project/data/raw'
os.makedirs(DATA_DIR, exist_ok=True)

_ = datasets.EMNIST(root=DATA_DIR, split='balanced', train=True,
                    download=True, transform=transforms.ToTensor())
_ = datasets.EMNIST(root=DATA_DIR, split='balanced', train=False,
                    download=True, transform=transforms.ToTensor())
print('✓ EMNIST готов')

## 5. Запуск финального эксперимента

In [ ]:
import subprocess, time
from datetime import datetime

exp_name = 'v5_FINAL_50ep_mixup02'

cmd = [
    'python', '-m', 'src.training.train',
    '--model',         'improved_cnn_v5',
    '--experiment',    exp_name,
    '--epochs',        '50',
    '--batch_size',    '64',
    '--lr',            '0.001',
    '--weight_decay',  '0.0001',
    '--augmentation',  'light',
    '--scheduler',     'cosine',
    '--split',         'balanced',
    '--seed',          '42',
    '--num_workers',   '2',
    '--mixup_alpha',   '0.2',     # ← главное изменение
]

print(f'Запуск: {exp_name}')
print(f'Старт: {datetime.now().strftime("%H:%M:%S")}')
print('=' * 70)

t0 = time.time()
result = subprocess.run(cmd, capture_output=False, text=True)
elapsed = (time.time() - t0) / 60

print('=' * 70)
print(f'Статус: {"OK" if result.returncode == 0 else "FAILED"} | Время: {elapsed:.1f} мин')

if result.returncode == 0:
    # Покажем финальный результат
    import glob
    import pandas as pd
    dirs = sorted(glob.glob(f'results/*{exp_name}*'),
                  key=os.path.getmtime, reverse=True)
    if dirs:
        df = pd.read_csv(f'{dirs[0]}/metrics/training_metrics.csv')
        print(f'\n📊 Лучшая точность: {df["test_acc"].max():.2f}%')
        print(f'   На эпохе: {int(df.loc[df["test_acc"].idxmax(), "epoch"])}')
        print(f'   Финальная (50 эпоха): {df["test_acc"].iloc[-1]:.2f}%')
        print(f'\nСравнение:')
        print(f'   v5_light_cosine (25 эп, без Mixup):  91.15%')
        print(f'   v5 + 50 эп + Mixup α=0.2:            {df["test_acc"].max():.2f}%')
        print(f'   Прирост:                            +{df["test_acc"].max() - 91.15:.2f} п.п.')

## 6. Архивация и сохранение

In [ ]:
# Полный архив
!cd /kaggle/working/emnist-project && \
  tar -czvf /kaggle/working/v5_final_full.tar.gz \
      results/*v5_FINAL* 2>&1 | tail -5

# Лёгкий для Claude
!cd /kaggle/working/emnist-project && \
  tar --exclude='*.pth' --exclude='*.pt' \
      --exclude='events.out.tfevents.*' --exclude='tensorboard' \
      -czvf /kaggle/working/v5_final_light.tar.gz \
      results/*v5_FINAL* 2>&1 | tail -5

!ls -lh /kaggle/working/v5_final_*.tar.gz
print('\n✓ Архивы доступны в Output справа')